## Experiments

- remove backgrounds
- test multiple seam line views for same poses

In [1]:
# CLO Viewpoints

import numpy as np
from dataclasses import dataclass

@dataclass
class CloCamera :
    name: str = None
    cam2world: np.ndarray = None
    fov: float = None
    width: int = None
    height: int = None
    
camera_info = {
    "Custom_View_1": CloCamera(
        name = "Custom_View_1",
        cam2world = np.array([
            [1, 0, 0, -9.2e-05],
            [0, 1, 0, -896.122],
            [0, 0, 1, -7998.56],
            [0, 0, 0, 1]
        ]),
        fov = 15,
        width = 480,
        height = 640
    ),  
    "Custom_View_2": CloCamera(
        name = "Custom_View_2",
        cam2world = np.array([
            [0.707107, 0, 0.707107, 1.01581],
            [0, 1, 0, -896.122],
            [-0.707107, 0, 0.707107, -7998.98],
            [0, 0, 0, 1]
        ]),
        fov = 15,
        width = 480,
        height = 640
    ),
    "Custom_View_3": CloCamera(
        name = "Custom_View_3",
        cam2world = np.array([
            [0.707107, 0, -0.707107, -1.01581],
            [0, 1, 0, -896.122],
            [0.707107, 0, 0.707107, -7998.98],
            [0, 0, 0, 1]
        ]),
        fov = 15,
        width = 480,
        height = 640
    ),
    "Custom_View_4": CloCamera(
        name = "Custom_View_4",
        cam2world = np.array([
            [0, 0, 1, 1.44049],
            [0, 1, 0, -896.122],
            [-1, 0, 0, -8000],
            [0, 0, 0, 1]
        ]),
        fov = 15,
        width = 480,
        height = 640
    ),
    "Custom_View_5": CloCamera(
        name = "Custom_View_5",
        cam2world = np.array([
            [0, 0, -1, -1.44093],
            [0, 1, 0, -896.122],
            [1, 0, 0, -8000],
            [0, 0, 0, 1]
        ]),
        fov = 15,
        width = 480,
        height = 640
    ),
    "Custom_View_6": CloCamera(
        name = "Custom_View_6",
        cam2world = np.array([
            [-1, 0, 0, -0.000608],
            [0, 1, 0, -896.122],
            [0, 0, -1, -8001.44],
            [0, 0, 0, 1]
        ]),
        fov = 15,
        width = 480,
        height = 640
    ),
    "Custom_View_7": CloCamera(
        name = "Custom_View_7",
        cam2world = np.array([
            [-0.677822, 0, -0.735226, -1.0604],
            [-0.051424, 0.997551, 0.047409, -893.857],
            [0.733425, 0.069943, -0.676162, -8063.65],
            [0, 0, 0, 1]
        ]),
        fov = 15,
        width = 480,
        height = 640
    ),
    "Custom_View_8": CloCamera(
        name = "Custom_View_8",
        cam2world = np.array([
            [-0.773786, 0, 0.633447, 0.911345],
            [0.035455, 0.998432, 0.043309, -894.654],
            [-0.632454, 0.055971, -0.772573, -8051.27],
            [0, 0, 0, 1]
        ]),
        fov = 15,
        width = 480,
        height = 640
    ),
    "Custom_View_9": CloCamera(
        name = "Custom_View_9",
        cam2world = np.array([
            [0.847255, 0, 0.531186, 0.765239],
            [0.223366, 0.907291, -0.356274, -813.553],
            [-0.48194, 0.420503, 0.768707, -8375.71],
            [0, 0, 0, 1]
        ]),
        fov = 15,
        width = 480,
        height = 640
    ),
    "Custom_View_10": CloCamera(
        name = "Custom_View_10",
        cam2world = np.array([
            [0.716937, 0, -0.697138, -1.0054],
            [-0.275326, 0.918708, -0.283145, -823.685],
            [0.640466, 0.394938, 0.658656, -8352.96],
            [0, 0, 0, 1]
        ]),
        fov = 15,
        width = 480,
        height = 640
    )   
}
for camera in camera_info.values() :
    camera.cam2world = np.linalg.inv(camera.cam2world)

In [2]:
import os, sys
import json
from pathlib import Path
import trimesh
import xml.etree.ElementTree as ET

from analysis_utils import visualize_meshes_plotly

@dataclass
class SeamDressScene :
    garment_dir: str = None
    garment_version: str = None
    
    spec_path_list: list = None
    stitch_dict_list: list = None
    
    pose_list: list = None
    mesh_path_list: list = None
    meta_path_list: list = None
    mesh_list: list = None
    
    rendered_path_list: list = None
    view_list: list = None

    def __post_init__(self, garment_version = "01"):
        self.garment_id_list = str(Path(self.garment_dir).name).split("__")[::2]
        self.spec_path_list = list(map(
            lambda gid : str(Path(self.garment_dir) / f"{gid}__{garment_version}__specification.json"),
            self.garment_id_list,
        ))
        self.stitch_dict_list = []
        for spec_path in self.spec_path_list:
            with open(spec_path, "r") as f:
                spec = json.load(f)
            stitch_dict = {}
            for idx, stitch in enumerate(spec["pattern"]["stitches"]):
                stitch_dict[idx] = stitch
            self.stitch_dict_list.append(stitch_dict)
            
        self.stitch_dict_dict = {}
        for garment_id, spec_path in zip(self.garment_id_list, self.spec_path_list) :
            with open(spec_path, "r") as f :
                spec = json.load(f)
            stitch_dict = {}
            for idx, stitch in enumerate(spec["pattern"]["stitches"]) :
                stitch_dict[idx] = stitch
            self.stitch_dict_dict[garment_id] = stitch_dict
        
        self.garment_count = len(self.spec_path_list)
        
        self.mesh_path_list = sorted(glob(str(Path(self.garment_dir)/"*.obj")))
        self.pose_list = list(map(lambda x: Path(x).stem, self.mesh_path_list))
        self.meta_path_list = list(map(
            lambda x : str(Path(self.garment_dir) / f"{x}_meta_data.xml"),
            self.pose_list
        ))
        
        self.rendered_path_list = list(filter(
            lambda x : "__" in Path(x).stem,
            glob(str(Path(self.garment_dir) / "*.png"))
        ))
        self.view_list = list(map(lambda x: Path(x).stem, self.rendered_path_list))
        
    def read_mesh(self):
        self.mesh_scene_list = list(map(
            lambda x: trimesh.load(x),
            self.mesh_path_list
        ))
        # self.mesh_dict = dict(zip(self.pose_list, self.mesh_list))
        
        # key : pose name, value : body mesh
        self.body_mesh_dict = {}
        self.garment_mesh_dict_dict = {}
        for pose_name, mesh_scene in zip(self.pose_list, self.mesh_scene_list) :
            mesh_list = []
            for geometry in mesh_scene.geometry.values() :
                if isinstance(geometry, trimesh.Trimesh):
                    mesh_list.append(geometry)
            self.body_mesh_dict[pose_name] = trimesh.util.concatenate(mesh_list[:-self.garment_count])
            self.garment_mesh_dict_dict[pose_name] = dict(zip(
                self.garment_id_list,
                mesh_list[-1:-self.garment_count-1:-1]
            ))
        
        self._get_stitch_vert()
        
    def _get_stitch_vert(self) :
        """
        Set below member variables.
        - self.stitch_vert_mask_dict_list
            - list for each garment, dict containing stitch vertex mask for each stitch
        - self.stitch_vert_idx_arr_list
        - self.stitch_vert_idx_arr_dict_list
        """
        
        # CLO saved metatdata does not distinguish stitches between garments
        raw_stitch_vert_idx_list_list = []
        
        tree = ET.parse(self.meta_path_list[0])
        root = tree.getroot()
        seam_line_pair_list = root.find("SeamLinePairList")
        for pair in seam_line_pair_list.findall("SeamLinePair") :
            seam_lines = pair.findall("SeamLine")
            if len(seam_lines) >= 2 :
                first_indexes_str = seam_lines[0].get("MeshPointIndexes")
                second_indexes_str = seam_lines[1].get("MeshPointIndexes")
                first_list = list(map(lambda x: int(x)-1, first_indexes_str.split("/")))
                second_list = list(map(lambda x: int(x)-1, second_indexes_str.split("/")))
                raw_stitch_vert_idx_list_list.append(first_list + second_list)
            else :
                print("SeamLinePair has less than 2 SeamLine")
        
        stitch_vert_idx_arr_list_dict = {}
        stitch_idx_accum = 0
        vert_count_accum = 0
        for (garment_id, stitch_dict) in self.stitch_dict_dict.items() :
            garment_mesh = list(self.garment_mesh_dict_dict.values())[0][garment_id]
            vert_count = garment_mesh.vertices.shape[0]
            
            stitch_vert_idx_arr_list = []
            for _ in stitch_dict.keys() :
                stitch_vert_idx_arr_list.append(
                    list(map(
                        lambda x: x-vert_count_accum,
                        raw_stitch_vert_idx_list_list[stitch_idx_accum]
                    ))
                )
                stitch_idx_accum += 1
            stitch_vert_idx_arr_list_dict[garment_id] = stitch_vert_idx_arr_list
            vert_count_accum += vert_count
        
        self.stitch_vert_idx_arr_list_dict = stitch_vert_idx_arr_list_dict
        
        self.stitch_vert_mask_dict_dict = {}
        for (garment_id, stitch_vert_idx_arr_list) in stitch_vert_idx_arr_list_dict.items() :
            garment_mesh = list(self.garment_mesh_dict_dict.values())[0][garment_id]
            vert_count = garment_mesh.vertices.shape[0]
            
            self.stitch_vert_mask_dict_dict[garment_id] = {}
            for (stitch_idx, stitch_vert_idx_arr) in enumerate(stitch_vert_idx_arr_list) :
                self.stitch_vert_mask_dict_dict[garment_id][stitch_idx] = np.zeros(
                    vert_count, dtype=bool
                )
                self.stitch_vert_mask_dict_dict[garment_id][stitch_idx][stitch_vert_idx_arr] = True
        
        # dict per garment, dict per stitch, array of seam line vertex idx, ordered so that it follows seam line linearly
        # vertex idx is already sorted linearly. just need to remove overlapping vertices
        # In the case of stitch between two garments, every seam vertex is duplicated,
        self.seam_line_vert_idx_arr_dict_dict = {}
        for garment_id, stitch_vert_idx_arr_list in stitch_vert_idx_arr_list_dict.items() :
            self.seam_line_vert_idx_arr_dict_dict[garment_id] = {}
            
            garment_mesh = list(self.garment_mesh_dict_dict.values())[0][garment_id]
            vert_count = garment_mesh.vertices.shape[0]
            
            for stch_idx, stch_vert_idx_arr in enumerate(stitch_vert_idx_arr_list) :
                seam_line_vert_idx_list = []
                for vert_idx in stch_vert_idx_arr :
                    
                    for i in seam_line_vert_idx_list :
                        if (garment_mesh.vertices[vert_idx] == garment_mesh.vertices[i]).all() :
                            break
                    else :
                        seam_line_vert_idx_list.append(vert_idx)

                self.seam_line_vert_idx_arr_dict_dict[garment_id][stch_idx] = np.array(seam_line_vert_idx_list)

[Line(start=(-54.7448-1465.06j), end=(-54.7448-838.648j)), CubicBezier(start=(-54.7448-838.648j), control1=(-54.7448-765.201j), control2=(-51.5242-692.01j), end=(-64.1571-619.076j)), Line(start=(-64.1571-619.076j), end=(-134.807-619.076j)), Line(start=(-134.807-619.076j), end=(-155.083-776.717j)), Line(start=(-155.083-776.717j), end=(-167.014-619.076j)), Line(start=(-167.014-619.076j), end=(-205.762-619.076j)), Line(start=(-205.762-619.076j), end=(-217.693-794.328j)), Line(start=(-217.693-794.328j), end=(-229.624-619.076j)), Line(start=(-229.624-619.076j), end=(-287.746-619.076j)), Line(start=(-287.746-619.076j), end=(-287.746-658.776j)), CubicBezier(start=(-287.746-658.776j), control1=(-273.489-849.437j), control2=(-308.664-913.564j), end=(-350.502-913.564j)), CubicBezier(start=(-350.502-913.564j), control1=(-339.169-1023.62j), control2=(-333.463-1198.29j), end=(-333.463-1455.9j)), Line(start=(-333.463-1455.9j), end=(-54.7448-1465.06j))]
[Line(start=(308.088-1336.48j), end=(550.556-13

In [3]:

import random
import matplotlib.pyplot as plt

In [4]:
import os, sys
from glob import glob
import random

garment_dir_list = sorted(glob(os.path.join("..", "SAMPLE_DATA", "GCD__GOOD", "*")))

In [5]:
IDX = 0
IDX = 1

scene = SeamDressScene(Path(garment_dir_list[IDX]))

In [43]:
scene.read_mesh()

IndexError: index -5573 is out of bounds for axis 0 with size 4511

In [6]:
rendered_name = random.choice(scene.rendered_path_list)

body_mesh = scene.body_mesh_dict[pose_name]
garment_mesh_dict = scene.garment_mesh_dict_dict[pose_name]

plt.imshow(plt.imread(rendered_name))
plt.show()

AttributeError: 'SeamDressScene' object has no attribute 'body_mesh_dict'

In [40]:
scene.garment_mesh_dict_dict

{'Custom_View_10__FV2_07': {'rand_2WFM2BO7V4': <trimesh.Trimesh(vertices.shape=(11073, 3), faces.shape=(19928, 3))>},
 'Custom_View_1__Dynamic': {'rand_2WFM2BO7V4': <trimesh.Trimesh(vertices.shape=(11073, 3), faces.shape=(19928, 3))>},
 'Custom_View_2__random': {'rand_2WFM2BO7V4': <trimesh.Trimesh(vertices.shape=(11059, 3), faces.shape=(19952, 3))>},
 'Custom_View_3__Dynamic_2': {'rand_2WFM2BO7V4': <trimesh.Trimesh(vertices.shape=(11073, 3), faces.shape=(19928, 3))>},
 'Custom_View_4__Dynamic_2': {'rand_2WFM2BO7V4': <trimesh.Trimesh(vertices.shape=(11059, 3), faces.shape=(19952, 3))>},
 'Custom_View_5__random': {'rand_2WFM2BO7V4': <trimesh.Trimesh(vertices.shape=(11073, 3), faces.shape=(19928, 3))>},
 'Custom_View_6__Dynamic': {'rand_2WFM2BO7V4': <trimesh.Trimesh(vertices.shape=(11759, 3), faces.shape=(21104, 3))>},
 'Custom_View_7__FV2_05': {'rand_2WFM2BO7V4': <trimesh.Trimesh(vertices.shape=(11073, 3), faces.shape=(19928, 3))>},
 'Custom_View_8__FV2_07': {'rand_2WFM2BO7V4': <trimesh.

In [41]:
garment_dir_list[1]

'..\\SAMPLE_DATA\\GCD__GOOD\\rand_300PM233IZ__01__rand_1DK7P8PVXM__01'

In [7]:
IDX = 2
GARMENT_DIR = garment_dir_list[IDX]
rendered_name_list = sorted(glob(os.path.join(GARMENT_DIR, "Custom*.png")))
mesh_name_list     = sorted(glob(os.path.join(GARMENT_DIR, "*.obj")))
metadata_name_list = sorted(glob(os.path.join(GARMENT_DIR, "*.xml")))

assert list(map(
    lambda x : Path(x).stem[:-2], rendered_name_list
)) == list(map(
    lambda x : Path(x).stem, mesh_name_list
)) and list(map(
    lambda x : Path(x).stem, mesh_name_list
)) == list(map(
    lambda x : Path(x).stem[:-10], metadata_name_list
)), "rendered_name_list, mesh_name_list, metadata_name_list are not consistent"


DATA_IDX = 0
rendered_name = rendered_name_list[DATA_IDX]
mesh_name     = mesh_name_list[DATA_IDX]
metadata_name = metadata_name_list[DATA_IDX]


In [8]:
mesh_scene = trimesh.load(mesh_name)



In [9]:
mesh_list = []
for geometry in mesh_scene.geometry.values() :
     if isinstance(geometry, trimesh.Trimesh) :
         mesh_list.append(geometry)

In [10]:
mesh_list

[<trimesh.Trimesh(vertices.shape=(5731, 3), faces.shape=(9592, 3))>,
 <trimesh.Trimesh(vertices.shape=(3580, 3), faces.shape=(2864, 3))>,
 <trimesh.Trimesh(vertices.shape=(596, 3), faces.shape=(1184, 3))>,
 <trimesh.Trimesh(vertices.shape=(6878, 3), faces.shape=(12950, 3))>,
 <trimesh.Trimesh(vertices.shape=(10736, 3), faces.shape=(20136, 3))>,
 <trimesh.Trimesh(vertices.shape=(7019, 3), faces.shape=(13548, 3))>,
 <trimesh.Trimesh(vertices.shape=(8187, 3), faces.shape=(16024, 3))>,
 <trimesh.Trimesh(vertices.shape=(1264, 3), faces.shape=(2393, 3))>,
 <trimesh.Trimesh(vertices.shape=(46211, 3), faces.shape=(61710, 3))>,
 <trimesh.Trimesh(vertices.shape=(4702, 3), faces.shape=(7788, 3))>,
 <trimesh.Trimesh(vertices.shape=(5065, 3), faces.shape=(8946, 3))>]